# 01-1 · Ingredients to dish taste vectors

| | |
|---|---|
| **What it does** | reads `ingredient-flavor.csv` (1000 ingredients) and every dish name in `data/Menu/*.json`, and infers a 6-D taste vector per dish by substring match |
| **Proves** | taste vectors are DERIVED from ingredients, never guessed by a model — the claim in the paper's Methods |
| **Ported from** | `potluck/backend/embeddings_creation/data_normalizer.py` (`infer_taste_from_text`, `diet_tag`) |
| **Eval surface** | taste coverage % — explains later why the fairness dial has limited room to move (see 03-aggregate/03) |

No API key needed. Fully deterministic — same input, same output, every run.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## The ingredient table — the only source of taste truth


In [3]:
# Step 2: load the ingredient -> flavor map
import csv
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    header = next(csv.reader(f))
print("CSV columns:", header)

flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        name = row.get("ingredient", "").strip().lower()
        if name:
            flavor_map[name] = {
                "sweet": float(row.get("sweet", 0) or 0), "salty": float(row.get("salty", 0) or 0),
                "sour": float(row.get("sour", 0) or 0), "bitter": float(row.get("bitter", 0) or 0),
                "umami": float(row.get("umami", 0) or 0), "spicy": float(row.get("spicy", 0) or 0),
            }
print("ingredients loaded:", len(flavor_map))


CSV columns: ['index', 'ingredient', 'spicy', 'sweet', 'umami', 'sour', 'salty']
ingredients loaded: 1000


> **Ceiling, worth stating up front.** The CSV header has no `bitter` column — the
> five real columns are `spicy, sweet, umami, sour, salty`. Every `bitter` value
> in this paper is a structural zero, not a measurement. The taste space is
> effectively 5-D, not 6-D.


## Read the real menus


In [4]:
# Step 3: read every dish name out of the raw menu JSON files
menu_files = sorted((nbio.DATA / "Menu").glob("*.json"))
print(f"menu files: {len(menu_files)}")

def iter_dish_names(node, path=None):
    path = path or []
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip()
            return
        for k, v in node.items():
            yield from iter_dish_names(v, path + [k])
    elif isinstance(node, list):
        for v in node:
            yield from iter_dish_names(v, path)

all_dishes = []
for f in menu_files:
    payload = json.loads(f.read_text())
    names = [n for n in iter_dish_names(payload) if n]
    all_dishes.extend(names)
    print(f"   {f.stem[:44]:<46} dishes={len(names):>4}")
print("TOTAL dishes:", len(all_dishes))


menu files: 5
   Arabian Corner Restaurant - Skylounge          dishes=  60
   Casa.cafe                                      dishes= 129
   Delicia Multi Cuisine Restaurant               dishes= 155
   House of Chefs                                 dishes= 198
   Ministry of South Bar and Kitchen              dishes=  32
TOTAL dishes: 574


## Infer taste, one dish at a time


In [5]:
# Step 4: infer a taste vector per dish -- mean of every ingredient whose name
# appears as a substring of the dish name
def infer_taste(dish_name, flavor_map):
    matched = [flavor_map[ing] for ing in flavor_map if ing in dish_name.lower()]
    if not matched:
        return [0.0] * 6
    dims = nbio.TASTE_DIMS
    return [sum(m[d] for m in matched) / len(matched) for d in dims]

tasted = [(name, infer_taste(name, flavor_map)) for name in all_dishes]
nonzero = [t for t in tasted if sum(abs(x) for x in t[1]) > 0]
print(f"taste coverage: {len(nonzero)}/{len(tasted)} = {100*len(nonzero)/len(tasted):.1f}%")
print()
print("MATCHED example:")
name, vec = nonzero[0]
print(f"   {name}")
print(nbio.taste(vec))


taste coverage: 267/574 = 46.5%

MATCHED example:
   Rooz Bukhari Rice
    sweet #........... 0.10
    salty ............ 0.00
    sour  ............ 0.00
    bitte ............ 0.00
    umami #........... 0.10
    spicy ............ 0.00


In [6]:
# Step 5: the failure mode -- dishes with no substring match
zero = [t for t in tasted if sum(abs(x) for x in t[1]) == 0]
print(f"ZERO taste vector: {len(zero)} dishes, e.g.:")
for name, _ in zero[:6]:
    print("   ", name)


ZERO taste vector: 307 dishes, e.g.:
    Dajjaj Roast Mandi
    Juicy Mutton
    Juicy Mutton Extra Piece
    Nalli Barra Mandi
    Special Murgh Tandoori
    Fish Tikka Amritsari


### Reading it

Substring matching against an ingredient table cannot read transliterated names
like "Dajjaj Roast Mandi" (chicken) — the ingredient table has no entry that is a
substring of that dish name. This is the measured reason **PRD-4** exists: an
offline, cached, LLM-based ingredient inference pass would close this gap, and
the code already has a fallback for exactly this case
(`taste_service.infer_taste_from_groq`) that the deterministic path here cannot
use. The coverage number above is the input every later notebook inherits —
watch it, because a satisfaction score built on a zero vector is meaningless.

Next: **01-2** turns these dish vectors into one taste vector per restaurant.
